# Curvilinear Boundary Conditions

Curvilinear grids need two boundary treatments. Outer ghost zones receive physical boundary data, while coordinate-induced inner ghost zones are filled from in-domain points with parity signs set by the reference-metric basis.

[Index](../index.ipynb) | Previous: [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb) | Next: [GeneralRFM and Fisheye Coordinates](generalrfm_and_fisheye.ipynb)

## Why This Matters

Curvilinear grids have outer boundaries and coordinate-induced inner ghost zones. Boundary metadata tracks how scalar, vector, and tensor components behave near those zones.

## What You Will See

- Parity classes for gridfunction components.
- Spherical parity assignment data.
- Finite-difference offsets for boundary extrapolation.

## Table of Contents

1. [Runtime context and imports](#Runtime-context-and-imports)
2. [Outer and inner ghost zones](#Outer-and-inner-ghost-zones)
3. [Parity classes](#Parity-classes)
4. [Radiation finite differences](#Radiation-finite-differences)
5. [Generated-project placement](#Generated-project-placement)
6. [Related notebooks](#Related-notebooks)

## Runtime Context and Imports

The notebook uses the importable `nrpy` package and inspects the current BHaH curvilinear-boundary tools without generating C functions.

In [1]:
import importlib.util

import nrpy
import nrpy.params as par
from nrpy.infrastructures.BHaH.CurviBoundaryConditions import (
    apply_bcs_outerradiation_and_inner as outer_bcs,
)
from nrpy.infrastructures.BHaH.CurviBoundaryConditions import (
    bcstruct_set_up,
    register_all,
)


print("nrpy import succeeded")

required_capabilities = {
    "bcstruct_set_up": (
        bcstruct_set_up,
        ["parity_conditions_symbolic_dot_products"],
    ),
    "apply_bcs_outerradiation_and_inner": (
        outer_bcs,
        [
            "get_arb_offset_FD_coeffs_indices",
            "setup_Cfunction_r_and_partial_xi_partial_r_derivs",
            "register_CFunction_apply_bcs_outerradiation_and_inner",
        ],
    ),
    "register_all": (register_all, ["register_C_functions"]),
}

for label, (module, names) in required_capabilities.items():
    print(label)
    for name in names:
        print(" ", name, "available:", hasattr(module, name))

nrpy import succeeded
bcstruct_set_up
  parity_conditions_symbolic_dot_products available: True
apply_bcs_outerradiation_and_inner
  get_arb_offset_FD_coeffs_indices available: True
  setup_Cfunction_r_and_partial_xi_partial_r_derivs available: True
  register_CFunction_apply_bcs_outerradiation_and_inner available: True
register_all
  register_C_functions available: True


## Outer and Inner Ghost Zones

The boundary setup first classifies ghost-zone points. Points that map to distinct in-domain points through the reference-metric eigencoordinate map are inner-boundary points. Remaining ghost-zone points are outer-boundary points and are grouped by face and layer for the selected physical condition.

The generated runtime path is:

1. build the boundary data structure for each registered coordinate system;
2. apply outer radiation or extrapolation data to outer ghost zones;
3. fill inner ghost zones by copying from mapped in-domain points with parity factors.

## Parity Classes

NRPy stores ten parity classes: one scalar class, three vector or one-form component classes, and six symmetric rank-2 tensor component classes. The signs come from dot products between basis unit vectors at the ghost point and its mapped in-domain point.

In [2]:
parity_classes = [
    "scalar",
    "x0 vector or one-form component",
    "x1 vector or one-form component",
    "x2 vector or one-form component",
    "x0x0 symmetric rank-2 component",
    "x0x1 symmetric rank-2 component",
    "x0x2 symmetric rank-2 component",
    "x1x1 symmetric rank-2 component",
    "x1x2 symmetric rank-2 component",
    "x2x2 symmetric rank-2 component",
]

for parity_index, description in enumerate(parity_classes):
    print(f"{parity_index}: {description}")

0: scalar
1: x0 vector or one-form component
2: x1 vector or one-form component
3: x2 vector or one-form component
4: x0x0 symmetric rank-2 component
5: x0x1 symmetric rank-2 component
6: x0x2 symmetric rank-2 component
7: x1x1 symmetric rank-2 component
8: x1x2 symmetric rank-2 component
9: x2x2 symmetric rank-2 component


In [3]:
par.set_parval_from_str("fp_type", "double")
parity_code = bcstruct_set_up.parity_conditions_symbolic_dot_products("Spherical")
assignment_lines = [
    line.strip()
    for line in parity_code.splitlines()
    if line.strip().startswith("REAL_parity_array")
]

print("Spherical parity assignment count:", len(assignment_lines))
print("Selected assignments:")
for line in assignment_lines[:4] + assignment_lines[-3:]:
    print(" ", line)

Setting up reference_metric[Spherical]...


Spherical parity assignment count: 10
Selected assignments:
  REAL_parity_array[0] = 1;
  REAL_parity_array[1] = tmp4;
  REAL_parity_array[2] = tmp5;
  REAL_parity_array[3] = tmp6;
  REAL_parity_array[7] = ((tmp5)*(tmp5));
  REAL_parity_array[8] = tmp5*tmp6;
  REAL_parity_array[9] = ((tmp6)*(tmp6));


## Radiation Finite Differences

Outer radiation conditions need radial derivatives. On a curvilinear grid, NRPy combines reference-metric radial derivatives with finite-difference stencils that can shift toward available grid points near a boundary.

In [4]:
for fd_order in (2, 4):
    for offset in (0, 1):
        coeffs, indices = outer_bcs.get_arb_offset_FD_coeffs_indices(
            fd_order, offset, 1
        )
        print(f"FD order {fd_order}, offset {offset}")
        print("  offsets:", indices)
        print("  coefficients:", [str(coeff) for coeff in coeffs])

FD order 2, offset 0
  offsets: [-1, 0, 1]
  coefficients: ['-1/2', '0', '1/2']
FD order 2, offset 1
  offsets: [0, 1, 2]
  coefficients: ['-3/2', '2', '-1/2']
FD order 4, offset 0
  offsets: [-2, -1, 0, 1, 2]
  coefficients: ['1/12', '-2/3', '0', '2/3', '-1/12']
FD order 4, offset 1
  offsets: [-1, 0, 1, 2, 3]
  coefficients: ['-1/4', '-5/6', '3/2', '-1/2', '1/12']


## Generated-Project Placement

BHaH wave projects register boundary C functions after numerical grids and right-hand sides are registered. The right-hand side is then passed to Method-of-Lines timestepping with curvilinear boundary calls in the step update.

In [5]:
generator_name = "nrpy.examples.wave_equation_curvilinear"
generator_spec = importlib.util.find_spec(generator_name)
print("Related generator discovered:", generator_spec is not None)

call_sequence = [
    "BHaH.numerical_grids_and_timestep.register_CFunctions(..., enable_CurviBCs=True)",
    "BHaH.CurviBoundaryConditions.register_all.register_C_functions(...)",
    "apply_bcs_outerradiation_and_inner(...) after each radiation RHS update",
    "apply_bcs_outerextrap_and_inner(...) after each extrapolation RHS update",
]

for step_number, call in enumerate(call_sequence, start=1):
    print(f"{step_number}. {call}")

Related generator discovered: True
1. BHaH.numerical_grids_and_timestep.register_CFunctions(..., enable_CurviBCs=True)
2. BHaH.CurviBoundaryConditions.register_all.register_C_functions(...)
3. apply_bcs_outerradiation_and_inner(...) after each radiation RHS update
4. apply_bcs_outerextrap_and_inner(...) after each extrapolation RHS update


## Related Notebooks

[Curvilinear wave equation](../3-wave_equation/wave_equation_curvilinear.ipynb) shows the generated wave-equation workflow that uses these calls. [GeneralRFM and fisheye coordinates](generalrfm_and_fisheye.ipynb) extends the reference-metric path to provider-backed mapped coordinates.

## Where This Leads

- [Curvilinear Wave Equation](../3-wave_equation/wave_equation_curvilinear.ipynb) reviews the prerequisite step.
- [GeneralRFM and Fisheye Coordinates](generalrfm_and_fisheye.ipynb) continues the tutorial path.
- [Index](../index.ipynb) returns to the full tutorial spine.